In [ ]:
# Install required libraries
!pip install google-cloud-bigquery pandas db-dtypes --quiet

In [ ]:
# Import BigQuery client and set configuration variables
from google.cloud import bigquery

PROJECT_ID = "qwiklabs-gcp-00-871084f9eb9e"
DATASET_ID = "weather_gemini"
RAW_TABLE = "weather_data"
OUTPUT_TABLE = "weather_alerts"
MODEL_NAME = "gemini_weather_model"
LOCATION = "US"

client = bigquery.Client(project=PROJECT_ID)

In [ ]:
# Create BigQuery dataset
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
client.create_dataset(dataset_ref, exists_ok=True)

print(f"Dataset {DATASET_ID} ready")

Dataset weather_gemini ready


In [ ]:
# Load weather CSV into BigQuery
load_job = client.load_table_from_uri(
    "gs://labs.roitraining.com/data-to-ai-workshop/weather_data.csv",
    f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}",
    job_config=bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        autodetect=True,
        skip_leading_rows=1,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    ),
)

load_job.result()
print("Weather data loaded")

Weather data loaded


In [ ]:
# Preview the raw data
df = client.query(f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}` LIMIT 5").to_dataframe()
df

,date,city,state,temperature_f,wind_speed_mph,precipitation_in,barometric_pressure_inHg,humidity_percent,weather_condition
0,2025-02-21,Atlanta,GA,55.7,5.0,0.12,29.80,50.4,Cloudy
1,2025-02-26,Atlanta,GA,75.2,10.4,0.03,29.58,49.9,Cloudy
2,2025-03-01,Atlanta,GA,51.7,4.7,0.08,29.74,49.9,Cloudy
3,2025-03-05,Atlanta,GA,74.4,5.1,0.02,29.92,50.4,Cloudy
4,2025-03-10,Atlanta,GA,59.5,9.6,0.09,29.67,57.2,Cloudy


In [ ]:
# Verify dataset and raw table are accessible
client.query(f"SELECT COUNT(*) AS total FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}`").to_dataframe()

,total
0,300


In [ ]:
# Show exact column names from the loaded table
client.query(f"""
SELECT column_name, data_type
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = '{RAW_TABLE}'
ORDER BY ordinal_position
""").to_dataframe()

,column_name,data_type
0,date,DATE
1,city,STRING
2,state,STRING
3,temperature_f,FLOAT64
4,wind_speed_mph,FLOAT64
5,precipitation_in,FLOAT64
6,barometric_pressure_inHg,FLOAT64
7,humidity_percent,FLOAT64
8,weather_condition,STRING


In [ ]:
# Create the Gemini remote model
create_model_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_weather_model`
REMOTE WITH CONNECTION `projects/qwiklabs-gcp-00-871084f9eb9e/locations/us/connections/gemini_conn`
OPTIONS (
  ENDPOINT = 'gemini-2.5-flash'
)
"""
client.query(create_model_sql).result()
print("Gemini model created successfully.")

Gemini model created successfully.


In [ ]:
# Generate weather reports from the table
generate_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.weather_reports` AS
SELECT
  date,
  city,
  state,
  temperature_f,
  wind_speed_mph,
  precipitation_in,
  barometric_pressure_inHg,
  humidity_percent,
  weather_condition,
  ml_generate_text_llm_result AS weather_report
FROM ML.GENERATE_TEXT(
  MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_weather_model`,
  (
    SELECT
      CONCAT(
        'Write a short weather report or warning based on this weather data. ',
        'Date: ', CAST(date AS STRING), '. ',
        'City: ', city, ', ', state, '. ',
        'Temperature: ', CAST(temperature_f AS STRING), ' F. ',
        'Wind speed: ', CAST(wind_speed_mph AS STRING), ' mph. ',
        'Precipitation: ', CAST(precipitation_in AS STRING), ' inches. ',
        'Barometric pressure: ', CAST(barometric_pressure_inHg AS STRING), ' inHg. ',
        'Humidity: ', CAST(humidity_percent AS STRING), ' percent. ',
        'Condition: ', weather_condition, '. ',
        'Respond in 1-2 sentences.'
      ) AS prompt,
      date,
      city,
      state,
      temperature_f,
      wind_speed_mph,
      precipitation_in,
      barometric_pressure_inHg,
      humidity_percent,
      weather_condition
    FROM `{PROJECT_ID}.{DATASET_ID}.weather_data`
  ),
  STRUCT(
    0.2 AS temperature,
    128 AS max_output_tokens,
    TRUE AS flatten_json_output
  )
);
"""
client.query(generate_sql).result()
print("weather_reports table created successfully.")

weather_reports table created successfully.


In [ ]:
# Verify the generated weather reports table
verify_sql = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.weather_reports`
LIMIT 10
"""
client.query(verify_sql).to_dataframe()

,date,city,state,temperature_f,wind_speed_mph,precipitation_in,barometric_pressure_inHg,humidity_percent,weather_condition,weather_report
0,2025-03-10,Atlanta,GA,59.5,9.6,0.09,29.67,57.2,Cloudy,"**Weather Report: Atlanta, GA - March 10, 2025..."
1,2025-03-01,Atlanta,GA,51.7,4.7,0.08,29.74,49.9,Cloudy,"Good morning, Atlanta! Expect a cloudy start t..."
2,2025-03-05,Atlanta,GA,74.4,5.1,0.02,29.92,50.4,Cloudy,"Atlanta, GA can expect a mild and cloudy day o..."
3,2025-02-26,Atlanta,GA,75.2,10.4,0.03,29.58,49.9,Cloudy,"Atlanta, expect a mild and cloudy day today, F..."
4,2025-03-14,Atlanta,GA,71.7,7.2,0.18,29.92,55.3,Cloudy,"Atlanta, GA can expect a cloudy day on March 1..."
5,2025-02-21,Atlanta,GA,55.7,5.0,0.12,29.80,50.4,Cloudy,"**Atlanta Weather Warning - February 21, 2025:..."
6,2025-02-19,Boston,MA,61.7,3.9,0.11,29.62,54.1,Cloudy,"Boston, MA can expect a mild and cloudy day to..."
7,2025-03-19,Boston,MA,60.7,6.4,0.04,29.83,49.4,Cloudy,"Boston can expect a cloudy day today, March 19..."
8,2025-03-09,Boston,MA,76.7,4.3,0.09,29.52,40.9,Cloudy,Boston is experiencing unseasonably warm and c...
9,2025-03-13,Boston,MA,71.9,9.8,0.16,29.99,42.3,Cloudy,"Good morning, Boston! Expect a mild and cloudy..."
